# Math Foundations Grand Solution — The Perfect Knuckleball Free Kick

> **Consolidated Notebook:** This notebook brings together all code examples from the Math Under the Hood track into a single executable demonstration. It shows the complete mathematical progression from linear approximations to full multi-parameter optimization under uncertainty.

**The Challenge:** Score a knuckleball free kick from 20m that:
1. ✅ Clears a 1.8m wall at 9.15m
2. ✅ Dips under a 2.44m crossbar
3. ✅ Beats the goalkeeper's reaction time

**Mathematical Progression:**
- Ch.1: Linear approximation → Predict first 0.1s only
- Ch.2: Polynomial features → Model full parabola
- Ch.3: Calculus → Find apex, verify constraints
- Ch.4: Gradient descent → Optimize single parameter
- Ch.5: Matrix operations → Handle 8+ features simultaneously
- Ch.6: Chain rule + gradients → Optimize ALL parameters at once
- Ch.7: Probability & MLE → Handle striker fatigue, wind variance

## Setup — Import Required Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import Ridge
from scipy.stats import norm

# Set plotting style
plt.style.use('dark_background')
plt.rcParams['figure.facecolor'] = '#1a1a2e'
plt.rcParams['axes.facecolor'] = '#1a1a2e'

print("✅ Libraries imported successfully")

## Ch.1: Linear Algebra — First 0.1 Seconds

**What it unlocks:** Model the free kick's first 0.1 seconds with $h(t) = 6.5t + 0$

**Key concept:** A line $y = wx + b$ is a two-parameter object. The weight $w$ (slope) tilts it, the bias $b$ (intercept) shifts it.

**Reality check:** By $t = 0.5$s, the linear model is 1.2m off — gravity bends the path into a curve.

In [ ]:
# Ch.1: Linear approximation (first 0.1 seconds)
def linear_height(t, w=6.5, b=0):
    """Linear model: h(t) = w*t + b"""
    return w * t + b

# Test early trajectory
t_early = np.linspace(0, 0.1, 20)
h_linear = linear_height(t_early)

print(f"Linear prediction at t=0.1s: {linear_height(0.1):.2f}m")
print(f"Slope (weight): 6.5 m/s, Intercept (bias): 0m")
print("⚠️ This breaks down after 0.1s — gravity is ignored!")

## Ch.2: Polynomial Features — Full Parabolic Trajectory

**What it unlocks:** Model the complete trajectory $h(t) = 6.5t - 4.905t^2$ (captures gravity)

**Key concept:** The curve $y = ax^2 + bx + c$ is non-linear in $x$ but *linear in the parameters* $(a, b, c)$. This **basis expansion** trick lets "linear" models fit curves.

**Breakthrough:** Can now predict at any time — $t = 0.6$s (wall) or $t = 1.2$s (goal).

In [ ]:
# Ch.2: Polynomial (parabolic) model
def parabolic_height(t, v0_y=6.5, g=9.81):
    """Full trajectory: h(t) = v0_y*t - 0.5*g*t^2"""
    return v0_y * t - 0.5 * g * t**2

# Full flight path
t_full = np.linspace(0, 1.4, 100)
h_parabola = parabolic_height(t_full)

# Key checkpoints
t_wall = 0.6  # Wall position
t_goal = 1.2  # Goal line

h_wall = parabolic_height(t_wall)
h_goal = parabolic_height(t_goal)

print(f"Height at wall (t={t_wall}s): {h_wall:.2f}m")
print(f"Height at goal (t={t_goal}s): {h_goal:.2f}m")
print("✅ Can now model full trajectory!")

## Ch.3: Calculus — Find the Apex and Verify Constraints

**What it unlocks:** Find the trajectory's peak by solving $h'(t) = 0$

**Key concept:** A derivative $f'(x)$ is the slope of a curve at one point. When $h'(t) = 0$, the ball has reached its apex.

**Breakthrough:** For the first time, we can *verify* constraints instead of just computing heights.

In [ ]:
# Ch.3: Find apex using calculus
def find_apex(v0_y=6.5, g=9.81):
    """Solve h'(t) = v0_y - g*t = 0 for t_peak"""
    t_peak = v0_y / g
    h_peak = parabolic_height(t_peak, v0_y, g)
    return t_peak, h_peak

t_peak, h_peak = find_apex()

# Verify constraints
wall_height = 1.8  # meters
crossbar_height = 2.44  # meters

wall_clearance = h_wall > wall_height
crossbar_clearance = h_goal < crossbar_height

print(f"\n🎯 Trajectory Analysis:")
print(f"  Peak time: {t_peak:.3f}s")
print(f"  Peak height: {h_peak:.3f}m")
print(f"\n🧱 Constraint #1 - Wall Clearance:")
print(f"  Height at wall: {h_wall:.2f}m > {wall_height}m? {'✅ YES' if wall_clearance else '❌ NO'}")
print(f"\n🎯 Constraint #2 - Crossbar Clearance:")
print(f"  Height at goal: {h_goal:.2f}m < {crossbar_height}m? {'✅ YES' if crossbar_clearance else '❌ NO'}")

## Ch.4: Gradient Descent — Optimize Launch Angle

**What it unlocks:** Find the launch angle $\theta^\star$ that maximizes range for fixed speed $v_0 = 25$ m/s

**Key concept:** To minimize $f(\theta)$, start somewhere, compute the slope $f'(\theta)$, step opposite to it, repeat. This is **gradient descent** — the algorithm that trains every neural network.

**Breakthrough:** Can now optimize a single parameter iteratively.

In [ ]:
# Ch.4: Gradient descent for optimal angle
def range_function(theta_deg, v0=25, g=9.81):
    """Compute horizontal range for given launch angle"""
    theta_rad = np.deg2rad(theta_deg)
    return (v0**2 * np.sin(2 * theta_rad)) / g

def gradient_range(theta_deg, v0=25, g=9.81):
    """Gradient of range w.r.t. angle (numerical approximation)"""
    epsilon = 0.01
    return (range_function(theta_deg + epsilon, v0, g) - range_function(theta_deg, v0, g)) / epsilon

# Gradient ascent (maximize range = minimize negative range)
theta = 20.0  # Starting angle
learning_rate = 0.15
history = [theta]

for step in range(20):
    grad = gradient_range(theta)
    theta = theta + learning_rate * grad  # Ascent (maximize)
    history.append(theta)
    if abs(grad) < 0.01:  # Convergence
        break

print(f"\n📈 Gradient Descent Results:")
print(f"  Starting angle: {history[0]:.1f}°")
print(f"  Optimal angle: {theta:.1f}°")
print(f"  Steps to convergence: {len(history)-1}")
print(f"  Maximum range: {range_function(theta):.1f}m")
print(f"  Theoretical optimum: 45° (for frictionless projectile)")

## Ch.5: Matrices — Multi-Feature Batch Operations

**What it unlocks:** Stack 500 free kicks × 8 features into design matrix $X \in \mathbb{R}^{500 \times 8}$

**Key concept:** A matrix $A \in \mathbb{R}^{m \times n}$ is a linear map. The product $A\mathbf{x}$ computes all outputs in one shot. The **normal equations** $(X^\top X)^{-1}X^\top \mathbf{y}$ solve least-squares regression analytically.

**Breakthrough:** Can now handle multiple features and batch predictions efficiently.

In [ ]:
# Ch.5: Matrix-based multi-feature regression
# Simulate dataset: 500 free kicks with 8 features
np.random.seed(42)
N = 500

# Features: launch_speed, angle, wind_speed, ball_pressure, kick_height, strike_zone_x, strike_zone_y, spin_rate
X = np.random.randn(N, 8)
X[:, 0] = np.random.uniform(20, 30, N)  # launch speed (m/s)
X[:, 1] = np.random.uniform(15, 60, N)  # angle (degrees)

# Target: distance traveled
true_weights = np.array([1.2, 0.5, -0.3, 0.1, 0.2, -0.1, 0.05, -0.02])
y = X @ true_weights + np.random.randn(N) * 0.5

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Add polynomial features (degree 2)
poly = PolynomialFeatures(degree=2, include_bias=True)
X_poly = poly.fit_transform(X_scaled)

print(f"\n📊 Multi-Feature Dataset:")
print(f"  Original features: {X.shape[1]} (8 features)")
print(f"  After polynomial expansion: {X_poly.shape[1]} features")
print(f"  Training samples: {N}")
print(f"  Feature matrix shape: {X_poly.shape}")
print(f"\n✅ Ready for batch predictions: ŷ = Xw (single matrix multiply!)")

## Pattern 1: Scale → Engineer → Fit

This is the production-ready workflow from Ch.1, Ch.2, and Ch.5 combined.

In [ ]:
# Pattern 1: Complete preprocessing + fitting pipeline
# (Already shown above, but explicitly stated as a reusable pattern)

# Fit with normal equations (for small datasets N < 10,000)
if N < 10_000:
    # Analytical solution: w = (X^T X)^{-1} X^T y
    XTX = X_poly.T @ X_poly
    XTy = X_poly.T @ y
    w_normal = np.linalg.solve(XTX, XTy)
    y_pred_normal = X_poly @ w_normal
    mse_normal = np.mean((y - y_pred_normal)**2)
    print(f"\n🔧 Normal Equations Solution:")
    print(f"  MSE: {mse_normal:.4f}")
    print(f"  Solved analytically (no iteration needed)")
else:
    print("\n⚠️ Dataset too large — would use gradient descent instead")

## Ch.6: Chain Rule & Backpropagation — Optimize All Parameters

**What it unlocks:** Simultaneously tune all 8+ parameters (angle, speed, height, wind compensation, etc.)

**Key concept:** The **matrix chain rule** composes gradients through layers: $\nabla_\mathbf{x} (g \circ f) = J_f^\top \nabla g$. This *is* backpropagation.

**Breakthrough:** Can now optimize MULTIPLE parameters at once — the full solution to the free kick challenge!

In [ ]:
# Ch.6: Backpropagation example - 2-layer network
def relu(x):
    return np.maximum(0, x)

def relu_grad(x):
    return (x > 0).astype(float)

def mse_loss(y_pred, y_true):
    return np.mean((y_pred - y_true)**2)

# Initialize small network for demonstration
np.random.seed(42)
input_dim = 8
hidden_dim = 16
output_dim = 1

W1 = np.random.randn(input_dim, hidden_dim) * 0.1
b1 = np.zeros(hidden_dim)
W2 = np.random.randn(hidden_dim, output_dim) * 0.1
b2 = np.zeros(output_dim)

# Forward pass
x_sample = X_scaled[0:1]  # Single sample
y_sample = y[0:1]

h1 = relu(x_sample @ W1 + b1)      # Ch.5: Matrix multiply
y_pred = h1 @ W2 + b2              # Ch.5: Matrix multiply
loss = mse_loss(y_pred, y_sample)  # Ch.7: Gaussian MLE

# Backward pass (chain rule)
dL_dy_pred = 2 * (y_pred - y_sample) / len(y_sample)
dL_dW2 = h1.T @ dL_dy_pred                           # Ch.6: Jacobian chain rule
dL_dh1 = dL_dy_pred @ W2.T                           # Ch.6: Reverse-mode autodiff
dL_dW1 = x_sample.T @ (dL_dh1 * relu_grad(x_sample @ W1 + b1))  # Ch.6: Chain rule

print(f"\n🔄 Backpropagation Demo:")
print(f"  Input shape: {x_sample.shape}")
print(f"  Hidden layer shape: {h1.shape}")
print(f"  Output shape: {y_pred.shape}")
print(f"  Loss (MSE): {loss:.6f}")
print(f"\n  Gradient shapes:")
print(f"    ∇W2: {dL_dW2.shape}")
print(f"    ∇W1: {dL_dW1.shape}")
print(f"\n✅ All gradients computed via chain rule — ready for optimization!")

## Pattern 3: Forward for Prediction, Backward for Optimization

This shows the duality at the heart of all neural network training.

In [ ]:
# Pattern 3: Training loop demonstration (1 step)
learning_rate = 0.01

# Update parameters (Ch.4: gradient descent)
W1_updated = W1 - learning_rate * dL_dW1
W2_updated = W2 - learning_rate * dL_dW2

# Verify loss decreases
h1_new = relu(x_sample @ W1_updated + b1)
y_pred_new = h1_new @ W2_updated + b2
loss_new = mse_loss(y_pred_new, y_sample)

print(f"\n📉 Gradient Descent Step:")
print(f"  Loss before: {loss:.6f}")
print(f"  Loss after:  {loss_new:.6f}")
print(f"  Improvement: {loss - loss_new:.6f}")
print(f"\n✅ Loss decreased — gradient descent is working!")

## Ch.7: Probability & MLE — Handle Uncertainty

**What it unlocks:** Model striker fatigue as $v_0 \sim \mathcal{N}(10, 0.3^2)$ instead of fixed $v_0 = 10$

**Key concept:** **Maximum Likelihood Estimation (MLE)** finds parameters that make observed data most probable. Minimizing MSE = maximizing likelihood under Gaussian noise.

**Breakthrough:** Can now reason about confidence intervals and success rates under realistic noise.

In [ ]:
# Ch.7: Uncertainty quantification
# Model striker fatigue and ball variance
v0_mean = 25.0  # m/s
v0_std = 0.5    # m/s
angle_mean = 30.0  # degrees
angle_std = 2.0    # degrees

# Simulate 1000 kicks with noise
np.random.seed(42)
n_simulations = 1000
v0_samples = np.random.normal(v0_mean, v0_std, n_simulations)
angle_samples = np.random.normal(angle_mean, angle_std, n_simulations)

# Compute range for each sample
ranges = np.array([range_function(angle, v0) for v0, angle in zip(v0_samples, angle_samples)])

# Statistics
range_mean = np.mean(ranges)
range_std = np.std(ranges)
range_ci_lower = range_mean - 2 * range_std
range_ci_upper = range_mean + 2 * range_std

# Success rate (range > 20m for goal)
success_rate = np.mean(ranges > 20)

print(f"\n🎲 Uncertainty Analysis (1000 kicks):")
print(f"  Launch speed: {v0_mean:.1f} ± {v0_std:.1f} m/s")
print(f"  Launch angle: {angle_mean:.1f}° ± {angle_std:.1f}°")
print(f"\n  Expected range: {range_mean:.2f}m")
print(f"  Standard deviation: {range_std:.2f}m")
print(f"  95% confidence interval: [{range_ci_lower:.2f}m, {range_ci_upper:.2f}m]")
print(f"  Success rate (range > 20m): {success_rate*100:.1f}%")
print(f"\n✅ Can now quantify confidence under realistic noise!")

## Pattern 4: Gaussian → MSE, Bernoulli → Cross-Entropy

**The connection between loss functions and probability distributions.**

In [ ]:
# Pattern 4: Loss functions as MLE
# Show that minimizing MSE = maximizing Gaussian likelihood

# Gaussian log-likelihood
def gaussian_log_likelihood(y_true, y_pred, sigma=1.0):
    """Log-likelihood under Gaussian noise model"""
    n = len(y_true)
    residuals = y_true - y_pred
    log_lik = -n/2 * np.log(2 * np.pi * sigma**2) - np.sum(residuals**2) / (2 * sigma**2)
    return log_lik

# Generate simple data
y_true_sample = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
y_pred_sample = np.array([1.1, 2.2, 2.8, 4.1, 4.9])

# Compute MSE and negative log-likelihood
mse = np.mean((y_true_sample - y_pred_sample)**2)
neg_log_lik = -gaussian_log_likelihood(y_true_sample, y_pred_sample, sigma=1.0)

print(f"\n📊 Loss Functions as MLE:")
print(f"  MSE: {mse:.6f}")
print(f"  Negative log-likelihood: {neg_log_lik:.6f}")
print(f"  Normalized NLL (÷ const): {neg_log_lik / len(y_true_sample):.6f}")
print(f"\n  Key insight: Minimizing MSE ≡ Maximizing Gaussian likelihood")
print(f"  This is why MSE is the default regression loss!")

## Complete Trajectory Visualization

Let's visualize the full trajectory with all constraints marked.

In [ ]:
# Complete visualization
fig, ax = plt.subplots(figsize=(12, 6))

# Time and horizontal position
t_viz = np.linspace(0, 1.4, 200)
v0 = 25  # m/s
angle = 30  # degrees
theta_rad = np.deg2rad(angle)

# Components
v0_x = v0 * np.cos(theta_rad)
v0_y = v0 * np.sin(theta_rad)

# Trajectory
x_pos = v0_x * t_viz
y_pos = v0_y * t_viz - 0.5 * 9.81 * t_viz**2

# Plot trajectory
ax.plot(x_pos, y_pos, 'cyan', linewidth=3, label='Ball trajectory')

# Mark constraints
wall_x = 9.15
goal_x = 20.0

# Wall
ax.axvline(wall_x, color='orange', linestyle='--', linewidth=2, alpha=0.7, label='Wall position (9.15m)')
ax.axhline(wall_height, color='red', linestyle='--', linewidth=2, alpha=0.7, label=f'Wall height ({wall_height}m)')

# Crossbar
ax.axvline(goal_x, color='yellow', linestyle='--', linewidth=2, alpha=0.7, label='Goal line (20m)')
ax.axhline(crossbar_height, color='magenta', linestyle='--', linewidth=2, alpha=0.7, label=f'Crossbar ({crossbar_height}m)')

# Apex
t_apex = v0_y / 9.81
x_apex = v0_x * t_apex
y_apex = v0_y * t_apex - 0.5 * 9.81 * t_apex**2
ax.plot(x_apex, y_apex, 'ro', markersize=10, label=f'Apex ({y_apex:.2f}m)')

ax.set_xlabel('Horizontal Distance (m)', fontsize=12)
ax.set_ylabel('Height (m)', fontsize=12)
ax.set_title('Complete Free Kick Trajectory with All Constraints', fontsize=14, fontweight='bold')
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 25)
ax.set_ylim(0, 4)

plt.tight_layout()
plt.show()

print("\n✅ Visualization complete!")

## Summary — Mathematical Toolkit Unlocked

**What we've built:**
- ✅ **Ch.1**: Linear models for early predictions
- ✅ **Ch.2**: Polynomial features for full trajectory
- ✅ **Ch.3**: Calculus to find apex and verify constraints
- ✅ **Ch.4**: Gradient descent for single-parameter optimization
- ✅ **Ch.5**: Matrix operations for multi-feature systems
- ✅ **Ch.6**: Backpropagation for multi-parameter optimization
- ✅ **Ch.7**: Probability for uncertainty quantification

**Production patterns demonstrated:**
1. Scale → Engineer → Fit (preprocessing pipeline)
2. Analytical when possible, iterative when necessary
3. Forward for prediction, backward for optimization
4. Loss functions as maximum likelihood estimators

**Next steps:**
- Proceed to **ML Track** (`notes/01-ml/`) to apply these foundations
- Every ML chapter builds directly on these seven mathematical tools
- Reference `grand_solution.md` for the complete conceptual narrative

**The complete solution:** We can now find launch parameters $(\theta, v_0)$ that satisfy all three constraints simultaneously and handle real-world uncertainty. Mission accomplished! 🎯